## 1. Installation

Run this cell to install all required dependencies.

In [24]:
# Install required packages
%pip install torch torchvision torchaudio
%pip install transformers sentencepiece accelerate
%pip install sentence-transformers
%pip install keybert
%pip install scipy numpy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Device Configuration

In [25]:
import torch
import json
import re
import warnings
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
from scipy.spatial.distance import cosine

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,  # Added for Qwen2
    pipeline
)
from sentence_transformers import SentenceTransformer, CrossEncoder
from keybert import KeyBERT

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

def get_device() -> torch.device:
    """
    Automatically detect and return the best available device.
    Priority: MPS (Apple Silicon) > CUDA > CPU
    """
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS (Apple Silicon) acceleration")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("Using CPU (No GPU acceleration available)")
    return device

# Initialize device
DEVICE = get_device()
print(f"Device: {DEVICE}")

Using MPS (Apple Silicon) acceleration
Device: mps


## 3. Data Classes for Structured Output

In [26]:
@dataclass
class MetricScores:
    """Container for all feature extraction scores."""
    semantic_score: float = 0.0
    coverage_score: float = 0.0
    missing_keywords: List[str] = field(default_factory=list)
    formality_score: float = 0.0
    grammar_score: float = 0.0
    logic_score: float = 0.0
    logic_details: Dict[str, float] = field(default_factory=dict)
    final_grade: float = 0.0  # Final grade on 0-100 scale


# Default weights for final grade calculation (must sum to 1.0)
DEFAULT_WEIGHTS = {
    "semantic": 0.20,
    "coverage": 0.20,
    "formality": 0.20,
    "grammar": 0.20,
    "logic": 0.20
}


def calculate_final_grade(
    metrics: 'MetricScores',
    weights: Dict[str, float] = None
) -> float:
    """
    Calculate final grade on 0-100 scale based on weighted combination of metrics.
    
    PRODUCTION-READY VERSION with:
    - Penalty for high contradiction (factual errors)
    - Proper handling of edge cases
    
    Args:
        metrics: MetricScores object containing all individual scores (0-1 scale)
        weights: Dictionary with weights for each criterion. Keys should be:
                 'semantic', 'coverage', 'formality', 'grammar', 'logic'
                 Weights should sum to 1.0. Default is 20% for each.
    
    Returns:
        Final grade on 0-100 scale
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS.copy()
    
    # Validate weights sum to 1.0 (with small tolerance for floating point)
    total_weight = sum(weights.values())
    if abs(total_weight - 1.0) > 0.001:
        # Normalize weights if they don't sum to 1.0
        weights = {k: v / total_weight for k, v in weights.items()}
    
    # Calculate weighted sum (all scores are 0-1, result is 0-100)
    base_grade = (
        metrics.semantic_score * weights.get("semantic", 0.20) +
        metrics.coverage_score * weights.get("coverage", 0.20) +
        metrics.formality_score * weights.get("formality", 0.20) +
        metrics.grammar_score * weights.get("grammar", 0.20) +
        metrics.logic_score * weights.get("logic", 0.20)
    ) * 100
    
    # PRODUCTION FIX: Apply penalty for high contradiction (factual/logical errors)
    # This ensures answers with wrong facts don't get inflated scores
    if metrics.logic_details:
        fwd_contradiction = metrics.logic_details.get("contradiction", 0)
        bwd_contradiction = metrics.logic_details.get("backward_contradiction", 0)
        max_contradiction = max(fwd_contradiction, bwd_contradiction)
        
        if max_contradiction > 0.90:
            # Severe factual error - cap grade at 40
            base_grade = min(base_grade, 40.0)
        elif max_contradiction > 0.70:
            # Significant factual error - cap grade at 55
            base_grade = min(base_grade, 55.0)
        elif max_contradiction > 0.50:
            # Moderate contradiction - apply percentage penalty
            penalty = (max_contradiction - 0.50) * 30  # Up to 15 point penalty
            base_grade = max(0, base_grade - penalty)
    
    # PRODUCTION FIX: Bonus for high-quality answers (semantic > 0.95, good grammar)
    # This helps "almost correct" answers get appropriate recognition
    if (metrics.semantic_score >= 0.95 and 
        metrics.grammar_score >= 0.90 and 
        metrics.logic_score >= 0.50 and
        len(metrics.missing_keywords) <= 1):
        # Near-perfect answer - ensure minimum grade of 85
        base_grade = max(base_grade, 85.0)
    
    return round(min(100.0, max(0.0, base_grade)), 2)


@dataclass
class LLMFeedback:
    """Container for LLM-generated feedback."""
    tags: List[str] = field(default_factory=list)
    explanation: str = ""
    suggestion: str = ""
    raw_response: str = ""
    parse_success: bool = False


@dataclass
class GradingResult:
    """Complete grading result containing all scores and feedback."""
    metrics: MetricScores = field(default_factory=MetricScores)
    feedback: LLMFeedback = field(default_factory=LLMFeedback)
    is_valid: bool = True
    skip_reason: Optional[str] = None
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary for JSON serialization."""
        return {
            "metrics": asdict(self.metrics),
            "feedback": asdict(self.feedback),
            "is_valid": self.is_valid,
            "skip_reason": self.skip_reason
        }
    
    def __repr__(self) -> str:
        return json.dumps(self.to_dict(), indent=2, ensure_ascii=False)

## 4. HybridASAGGrader Class

The main class implementing the hybrid grading architecture.

In [27]:
class HybridASAGGrader:
    """
    Hybrid Automated Short Answer Grading System - OPTIMIZED VERSION
    
    Uses specialized models for feature extraction and Qwen2.5-3B for reasoning.
    Optimized for Apple Silicon (MPS) with fallback to CUDA/CPU.
    
    Tag System (Hierarchical - mutually exclusive levels):
    ┌─────────────────────────────────────────────────────────┐
    │ CORRECTNESS LEVEL (pick one):                           │
    │   • Correct          - Fully correct answer             │
    │   • Partially Correct - On topic but incomplete         │
    │   • Incorrect        - Wrong/contradictory answer       │
    ├─────────────────────────────────────────────────────────┤
    │ ISSUE TAGS (can have multiple, only if not "Correct"):  │
    │   • Missing Concepts  - Key terms/ideas missing         │
    │   • Factual Error     - Contains wrong information      │
    │   • Logical Error     - Contradicts reference/logic     │
    │   • Vague Expression  - Too general, lacks specificity  │
    │   • Grammar Error     - Poor grammar/spelling           │
    │   • Off-Topic         - Doesn't address the question    │
    │   • Incomplete        - Answer cut off/too brief        │
    └─────────────────────────────────────────────────────────┘
    """
    
    # Model identifiers
    SEMANTIC_MODEL = "princeton-nlp/sup-simcse-roberta-large"
    KEYBERT_MODEL = "all-MiniLM-L6-v2"
    FORMALITY_MODEL = "cointegrated/roberta-base-formality"
    GRAMMAR_MODEL = "textattack/roberta-base-CoLA"
    LOGIC_MODEL = "cross-encoder/nli-deberta-v3-base"
    REASONING_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    
    # Tag definitions
    CORRECTNESS_TAGS = ["Correct", "Partially Correct", "Incorrect"]
    ISSUE_TAGS = ["Missing Concepts", "Factual Error", "Logical Error", 
                  "Vague Expression", "Grammar Error", "Off-Topic", "Incomplete"]
    ALL_TAGS = CORRECTNESS_TAGS + ISSUE_TAGS
    
    # Thresholds - FULLY OPTIMIZED
    MIN_WORDS_FOR_FULL_ANALYSIS = 5
    KEYWORD_SIMILARITY_THRESHOLD = 0.50  # Lowered for better synonym matching
    
    THRESHOLDS = {
        # Correctness thresholds
        "semantic_correct": 0.85,      # High semantic = correct candidate
        "semantic_partial": 0.55,      # Moderate semantic = partial
        "semantic_off_topic": 0.35,    # Low semantic = off-topic
        
        # Coverage thresholds
        "coverage_correct": 0.80,      # High coverage required for Correct
        "coverage_good": 0.60,         # Good coverage threshold
        "coverage_missing": 0.60,      # Below = Missing Concepts
        
        # Grammar threshold - LOWERED for poor grammar detection
        "grammar_good": 0.50,          # Lowered from 0.65
        "grammar_poor": 0.35,          # Below = definitely grammar issues
        
        # Logic thresholds - ADJUSTED for NLI quirks
        "logic_correct": 0.50,         # Lowered for Correct
        "logic_good": 0.40,            # Lowered for good logic
        "logic_error": 0.20,           # Lowered for error detection
        
        # Contradiction thresholds
        "contradiction_high": 0.60,    # Raised to reduce false positives
        "contradiction_moderate": 0.35,
        
        # Formality threshold
        "formality_poor": 0.10,        # Very informal writing
    }
    
    def __init__(self, device: torch.device = None, verbose: bool = True):
        """Initialize the grader with all required models."""
        self.device = device or get_device()
        self.verbose = verbose
        self._load_models()
    
    def _log(self, message: str):
        """Print message if verbose mode is enabled."""
        if self.verbose:
            print(message)
    
    def _load_models(self):
        """Load all required models with memory optimization."""
        self._log("\n" + "="*60)
        self._log("Loading Models...")
        self._log("="*60)
        
        # 1. Semantic Similarity Model
        self._log("\n[1/6] Loading Semantic Model (SimCSE)...")
        self.semantic_model = SentenceTransformer(self.SEMANTIC_MODEL)
        if self.device.type != "cpu":
            self.semantic_model = self.semantic_model.to(self.device)
        self._log("      Done: princeton-nlp/sup-simcse-roberta-large")
        
        # 2. KeyBERT
        self._log("\n[2/6] Loading KeyBERT Model...")
        self.keybert_model = KeyBERT(model=self.KEYBERT_MODEL)
        self._log("      Done: all-MiniLM-L6-v2 (KeyBERT backend)")
        
        # 3. Formality Classifier
        self._log("\n[3/6] Loading Formality Model...")
        self.formality_tokenizer = AutoTokenizer.from_pretrained(self.FORMALITY_MODEL)
        self.formality_model = AutoModelForSequenceClassification.from_pretrained(
            self.FORMALITY_MODEL
        ).to(self.device)
        self.formality_model.eval()
        self._log("      Done: cointegrated/roberta-base-formality")
        
        # 4. Grammar Classifier
        self._log("\n[4/6] Loading Grammar Model...")
        self.grammar_tokenizer = AutoTokenizer.from_pretrained(self.GRAMMAR_MODEL)
        self.grammar_model = AutoModelForSequenceClassification.from_pretrained(
            self.GRAMMAR_MODEL
        ).to(self.device)
        self.grammar_model.eval()
        self._log("      Done: textattack/roberta-base-CoLA")
        
        # 5. Logic/NLI CrossEncoder
        self._log("\n[5/6] Loading Logic Model (NLI)...")
        self.logic_model = CrossEncoder(self.LOGIC_MODEL)
        self._log("      Done: cross-encoder/nli-deberta-v3-base")
        
        # 6. Qwen2.5-3B for reasoning (optimized for MPS)
        self._log("\n[6/6] Loading Reasoning Model (Qwen2.5-3B-Instruct)...")
        self.reasoning_tokenizer = AutoTokenizer.from_pretrained(
            self.REASONING_MODEL,
            trust_remote_code=True
        )
        
        # Determine optimal dtype for device
        if self.device.type == "mps":
            # MPS works best with float16 for large models
            model_dtype = torch.float16
        elif self.device.type == "cuda":
            model_dtype = torch.float16
        else:
            model_dtype = torch.float32
        
        self.reasoning_model = AutoModelForCausalLM.from_pretrained(
            self.REASONING_MODEL,
            torch_dtype=model_dtype,
            trust_remote_code=True,
            low_cpu_mem_usage=True  # Better memory management for larger model
        ).to(self.device)
        self.reasoning_model.eval()
        self._log("      Done: Qwen/Qwen2.5-3B-Instruct")
        
        self._log("\n" + "="*60)
        self._log("All models loaded successfully!")
        self._log("="*60 + "\n")
    
    # =========================================================================
    # FEATURE EXTRACTION METHODS
    # =========================================================================
    
    def get_semantic_score(self, reference: str, student: str) -> float:
        """Calculate semantic similarity using SimCSE embeddings."""
        with torch.no_grad():
            embeddings = self.semantic_model.encode(
                [reference, student],
                convert_to_tensor=True,
                show_progress_bar=False
            )
            ref_emb = embeddings[0].cpu().numpy()
            stu_emb = embeddings[1].cpu().numpy()
            similarity = 1 - cosine(ref_emb, stu_emb)
            return float(max(0.0, min(1.0, similarity)))
    
    def get_keyword_coverage(
        self, 
        question: str, 
        reference: str, 
        student: str
    ) -> Tuple[float, List[str]]:
        """Calculate keyword coverage using KeyBERT + semantic matching."""
        context_doc = reference
        word_count = len(reference.split())
        num_keywords = max(3, min(8, word_count // 15 + 2))
        
        # Extract unigrams
        unigram_keywords = self.keybert_model.extract_keywords(
            context_doc,
            keyphrase_ngram_range=(1, 1),
            stop_words='english',
            top_n=num_keywords,
            use_mmr=True,
            diversity=0.5
        )
        
        # Extract bigrams (filtered)
        bigram_keywords = self.keybert_model.extract_keywords(
            context_doc,
            keyphrase_ngram_range=(2, 2),
            stop_words='english',
            top_n=max(2, num_keywords // 2),
            use_mmr=True,
            diversity=0.7
        )
        good_bigrams = [(kw, score) for kw, score in bigram_keywords if score > 0.35]
        
        all_keywords = unigram_keywords + good_bigrams
        if not all_keywords:
            return 1.0, []
        
        keyword_list = [kw[0] for kw in all_keywords if len(kw[0]) > 2]
        seen = set()
        keyword_list = [x for x in keyword_list if not (x.lower() in seen or seen.add(x.lower()))]
        
        if not keyword_list:
            return 1.0, []
        
        student_lower = student.lower()
        student_words = set(student_lower.split())
        
        with torch.no_grad():
            all_texts = keyword_list + [student]
            embeddings = self.semantic_model.encode(
                all_texts,
                convert_to_tensor=True,
                show_progress_bar=False
            )
            keyword_embeddings = embeddings[:-1].cpu().numpy()
            student_embedding = embeddings[-1].cpu().numpy()
        
        found_keywords = []
        missing_keywords = []
        
        for i, keyword in enumerate(keyword_list):
            keyword_lower = keyword.lower()
            
            # Method 1: Exact/substring match
            if keyword_lower in student_lower:
                found_keywords.append(keyword)
                continue
            
            # Method 2: Word-level match for unigrams
            if ' ' not in keyword and keyword_lower in student_words:
                found_keywords.append(keyword)
                continue
            
            # Method 3: Semantic similarity matching
            similarity = 1 - cosine(keyword_embeddings[i], student_embedding)
            if similarity >= self.KEYWORD_SIMILARITY_THRESHOLD:
                found_keywords.append(keyword)
            else:
                missing_keywords.append(keyword)
        
        coverage_score = len(found_keywords) / len(keyword_list) if keyword_list else 1.0
        return float(coverage_score), missing_keywords
    
    def get_formality_score(self, text: str) -> float:
        """Calculate formality score using RoBERTa classifier."""
        with torch.no_grad():
            inputs = self.formality_tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)
            outputs = self.formality_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            return float(probs[0][1].item())
    
    def get_grammar_score(self, text: str) -> float:
        """Calculate grammar acceptability score using CoLA model."""
        with torch.no_grad():
            inputs = self.grammar_tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)
            outputs = self.grammar_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            return float(probs[0][1].item())
    
    def get_logic_score(
        self, 
        reference: str, 
        student: str,
        grammar_score: float = None
    ) -> Tuple[float, Dict[str, float]]:
        """
        Calculate logical coherence using bidirectional NLI.
        
        IMPROVED: Considers grammar quality - poor grammar can confuse NLI models.
        """
        scores = self.logic_model.predict(
            [
                (reference, student),
                (student, reference)
            ],
            apply_softmax=True
        )
        
        forward_probs = scores[0]
        backward_probs = scores[1]
        
        prob_dict = {
            "contradiction": float(forward_probs[0]),
            "entailment": float(forward_probs[1]),
            "neutral": float(forward_probs[2]),
            "backward_entailment": float(backward_probs[1]),
            "backward_contradiction": float(backward_probs[0]),
            "backward_neutral": float(backward_probs[2])
        }
        
        fwd_contradiction = float(forward_probs[0])
        fwd_entailment = float(forward_probs[1])
        fwd_neutral = float(forward_probs[2])
        
        bwd_entailment = float(backward_probs[1])
        bwd_contradiction = float(backward_probs[0])
        bwd_neutral = float(backward_probs[2])
        
        max_contradiction = max(fwd_contradiction, bwd_contradiction)
        avg_entailment = (fwd_entailment + bwd_entailment) / 2
        avg_neutral = (fwd_neutral + bwd_neutral) / 2
        
        # Calculate base logic score
        if max_contradiction > self.THRESHOLDS["contradiction_high"]:
            logic_score = (1 - max_contradiction) * 0.15
        elif max_contradiction > self.THRESHOLDS["contradiction_moderate"]:
            base_score = avg_entailment
            penalty = max_contradiction * 0.3
            logic_score = base_score * (1 - penalty) + 0.25
        elif avg_neutral > 0.8:
            logic_score = 0.55 + (avg_entailment * 0.25) + ((1 - max_contradiction) * 0.15)
        else:
            entailment_score = (fwd_entailment * 0.55 + bwd_entailment * 0.35)
            neutral_bonus = min(fwd_neutral, bwd_neutral) * 0.15
            logic_score = entailment_score + neutral_bonus + 0.15
        
        # GRAMMAR ADJUSTMENT: If grammar is poor, NLI may be unreliable
        # Boost logic score slightly if grammar is the issue (not contradiction)
        if grammar_score is not None and grammar_score < self.THRESHOLDS["grammar_good"]:
            if max_contradiction < self.THRESHOLDS["contradiction_high"]:
                # Grammar issues may cause low entailment, not contradiction
                grammar_boost = (self.THRESHOLDS["grammar_good"] - grammar_score) * 0.3
                logic_score = min(1.0, logic_score + grammar_boost)
        
        logic_score = max(0.0, min(1.0, logic_score))
        return float(logic_score), prob_dict
    
    def get_simple_word_match_score(self, reference: str, student: str) -> float:
        """Simple word overlap score for very short answers."""
        ref_words = set(reference.lower().split())
        stu_words = set(student.lower().split())
        if not ref_words:
            return 0.0
        overlap = ref_words.intersection(stu_words)
        return len(overlap) / len(ref_words)
    
    # =========================================================================
    # TAG ASSIGNMENT LOGIC - FULLY OPTIMIZED
    # =========================================================================
    
    def _assign_tags(self, metrics: MetricScores) -> Tuple[List[str], str]:
        """
        Assign tags using optimized multi-factor scoring.
        
        Uses weighted combination of metrics for more accurate classification.
        IMPROVED: Adjusts semantic score when high contradiction is detected.
        """
        tags = []
        correctness = None
        
        # Get contradiction info
        fwd_contradiction = metrics.logic_details.get("contradiction", 0)
        bwd_contradiction = metrics.logic_details.get("backward_contradiction", 0)
        max_contradiction = max(fwd_contradiction, bwd_contradiction)
        has_high_contradiction = fwd_contradiction > self.THRESHOLDS["contradiction_high"] or \
                                 bwd_contradiction > self.THRESHOLDS["contradiction_high"]
        has_moderate_contradiction = fwd_contradiction > self.THRESHOLDS["contradiction_moderate"] or \
                                     bwd_contradiction > self.THRESHOLDS["contradiction_moderate"]
        
        # IMPROVED: Adjust semantic score when contradiction is high
        # This prevents high semantic score for factually incorrect answers
        adjusted_semantic = metrics.semantic_score
        if has_high_contradiction:
            # Significant penalty for contradictory answers
            adjusted_semantic = metrics.semantic_score * (1 - max_contradiction * 0.6)
        elif has_moderate_contradiction:
            # Moderate penalty
            adjusted_semantic = metrics.semantic_score * (1 - max_contradiction * 0.3)
        
        # Calculate composite score for correctness
        # Weight: semantic (0.45), coverage (0.20), logic (0.20), grammar (0.15)
        composite_score = (
            adjusted_semantic * 0.45 +
            metrics.coverage_score * 0.20 +
            metrics.logic_score * 0.20 +
            metrics.grammar_score * 0.15
        )
        
        # Check if grammar is the primary issue (high semantic but poor grammar)
        grammar_is_primary_issue = (
            metrics.semantic_score >= self.THRESHOLDS["semantic_partial"] and
            metrics.grammar_score < self.THRESHOLDS["grammar_good"] and
            not has_high_contradiction
        )
        
        # STEP 1: Determine correctness level (using adjusted_semantic for contradiction cases)
        if has_high_contradiction:
            correctness = "Incorrect"
        elif adjusted_semantic >= self.THRESHOLDS["semantic_correct"] and \
             metrics.logic_score >= self.THRESHOLDS["logic_correct"] and \
             not has_moderate_contradiction and \
             metrics.grammar_score >= self.THRESHOLDS["grammar_good"]:
            # High quality answer
            correctness = "Correct"
        elif adjusted_semantic >= self.THRESHOLDS["semantic_correct"] and \
             not has_moderate_contradiction and \
             composite_score >= 0.65:
            # Good overall with high semantic
            correctness = "Correct"
        elif adjusted_semantic >= self.THRESHOLDS["semantic_partial"] or composite_score >= 0.45:
            # Partial answer - semantic is decent OR composite is okay
            correctness = "Partially Correct"
        elif adjusted_semantic < self.THRESHOLDS["semantic_off_topic"]:
            correctness = "Incorrect"
        else:
            correctness = "Partially Correct"
        
        tags.append(correctness)
        
        # STEP 2: If "Correct", check for missing keywords - may need to downgrade
        if correctness == "Correct":
            # PRODUCTION FIX: Only downgrade if multiple keywords missing or coverage is very low
            # Allow 1 missing keyword for high semantic answers (synonyms/paraphrasing)
            num_missing = len(metrics.missing_keywords) if metrics.missing_keywords else 0
            should_downgrade = (
                (num_missing >= 2 and metrics.coverage_score < self.THRESHOLDS["coverage_correct"]) or
                (num_missing >= 3) or  # Too many missing regardless of coverage
                (metrics.coverage_score < 0.50)  # Very low coverage
            )
            
            if should_downgrade:
                # Downgrade to Partially Correct with Missing Concepts tag
                tags[0] = "Partially Correct"
                correctness = "Partially Correct"
                tags.append("Missing Concepts")
            return tags, correctness
        
        # STEP 3: Add issue tags (only for non-Correct)
        
        # Off-topic check (use raw semantic for off-topic, as contradiction != off-topic)
        if metrics.semantic_score < self.THRESHOLDS["semantic_off_topic"]:
            tags.append("Off-Topic")
        
        # Logical/Factual error check - only for clear contradictions
        if has_high_contradiction:
            tags.append("Logical Error")
            if metrics.semantic_score >= self.THRESHOLDS["semantic_off_topic"]:
                tags.append("Factual Error")
        elif metrics.logic_score < self.THRESHOLDS["logic_error"] and not grammar_is_primary_issue:
            if "Off-Topic" not in tags:
                tags.append("Factual Error")
        
        # Missing concepts check (triggers when coverage < threshold OR has 2+ missing keywords)
        if metrics.missing_keywords and (metrics.coverage_score < self.THRESHOLDS["coverage_missing"] or len(metrics.missing_keywords) >= 2):
            if "Missing Concepts" not in tags:
                tags.append("Missing Concepts")
        
        # Vague expression check (only if not already tagged with Missing Concepts)
        # Triggers when semantic is okay but coverage is low without specific missing keywords
        if adjusted_semantic >= self.THRESHOLDS["semantic_partial"] and \
           metrics.coverage_score < self.THRESHOLDS["coverage_good"] and \
           "Missing Concepts" not in tags and \
           "Grammar Error" not in tags and \
           not metrics.missing_keywords:
            tags.append("Vague Expression")
        
        # Grammar error check
        if metrics.grammar_score < self.THRESHOLDS["grammar_good"]:
            tags.append("Grammar Error")
        
        return tags, correctness
    
    # =========================================================================
    # REASONING LAYER (QWEN2) - OPTIMIZED PROMPTS
    # =========================================================================
    
    def _build_prompt(self, 
                      context: str,
                      question: str,
                      reference: str,
                      student: str,
                      metrics: MetricScores) -> str:
        """Build optimized prompt for Qwen2 with improved JSON formatting."""
        missing_list = ", ".join(metrics.missing_keywords[:3]) if metrics.missing_keywords else "None"
        
        # Pre-assign tags
        tags, correctness = self._assign_tags(metrics)
        tags_json = json.dumps(tags)  # Proper JSON array format
        
        # Build specific error analysis for better suggestions
        error_details = []
        if "Missing Concepts" in tags and metrics.missing_keywords:
            error_details.append(f"Missing key terms: {', '.join(metrics.missing_keywords[:3])}")
        if "Logical Error" in tags or "Factual Error" in tags:
            error_details.append("Contains factual/logical errors that contradict the reference")
        if "Vague Expression" in tags:
            error_details.append("Answer is too general, lacks specific details")
        if "Grammar Error" in tags:
            error_details.append("Has grammar/spelling issues")
        
        error_analysis = "; ".join(error_details) if error_details else "Minor issues only"
        
        # Improved Qwen2 prompt - request SPECIFIC suggestions
        prompt = f'''<|im_start|>system
You are a grading assistant. Provide specific, actionable feedback. Output ONLY valid JSON.
<|im_end|>
<|im_start|>user
Grade this student answer with SPECIFIC feedback.

Question: "{question[:100]}"
Reference Answer: "{reference[:150]}"
Student Answer: "{student[:150]}"

Analysis:
- Grade: {correctness}
- Issues: {error_analysis}
- Missing Keywords: {missing_list}

Requirements for your response:
1. explanation: Explain what is wrong or right, referencing specific parts of the student's answer
2. suggestion: Give SPECIFIC advice on how to improve, including what words/concepts to add or correct

Output JSON:
{{"tags": {tags_json}, "explanation": "specific explanation referencing student answer", "suggestion": "specific improvement advice"}}
<|im_end|>
<|im_start|>assistant
{{"tags": {tags_json}, "explanation": "'''
        return prompt
    
    def _parse_llm_response(self, response: str, metrics: MetricScores = None) -> LLMFeedback:
        """Parse JSON response with robust fallback and partial response reconstruction."""
        feedback = LLMFeedback(raw_response=response)
        
        try:
            response = response.strip()
            
            # Remove markdown code blocks
            for prefix in ["```json", "```"]:
                if response.startswith(prefix):
                    response = response[len(prefix):]
            if response.endswith("```"):
                response = response[:-3]
            
            response = response.strip()
            
            # CASE 1: Response starts with continuation (from our prompt ending with "explanation": ")
            # The response might look like: 'The student....", "suggestion": "Add..."}'
            if not response.startswith('{'):
                # Try to reconstruct JSON by prepending what we started with
                tags, correctness = self._assign_tags(metrics) if metrics else (["Partially Correct"], "Partially Correct")
                tags_json = json.dumps(tags)
                
                # Check if response has suggestion pattern
                if '"suggestion"' in response or "'suggestion'" in response:
                    # Reconstruct the full JSON
                    reconstructed = '{"tags": ' + tags_json + ', "explanation": "' + response
                    
                    # Try to parse reconstructed JSON
                    try:
                        # Clean up the reconstructed string
                        reconstructed = reconstructed.replace("'", '"')
                        reconstructed = re.sub(r',\s*}', '}', reconstructed)
                        
                        # Find the last valid }
                        last_brace = reconstructed.rfind('}')
                        if last_brace != -1:
                            reconstructed = reconstructed[:last_brace + 1]
                        
                        data = json.loads(reconstructed)
                        feedback.tags = self._validate_tags(data.get("tags", []), metrics)
                        feedback.explanation = str(data.get("explanation", ""))[:500]
                        feedback.suggestion = str(data.get("suggestion", ""))[:400]
                        feedback.parse_success = True
                        return feedback
                    except:
                        pass
                
                # CASE 1b: Try to extract explanation and suggestion using regex
                exp_match = re.search(r'^([^"]+)', response)
                sug_match = re.search(r'"suggestion"\s*:\s*"([^"]+)"', response)
                
                if exp_match:
                    explanation = exp_match.group(1).strip().rstrip('",')
                    suggestion = sug_match.group(1) if sug_match else ""
                    
                    feedback.tags = self._validate_tags([], metrics)
                    feedback.explanation = explanation[:500]
                    feedback.suggestion = suggestion[:400]
                    feedback.parse_success = True
                    return feedback
            
            # CASE 2: Standard JSON parsing
            start_idx = response.find('{')
            end_idx = response.rfind('}')
            
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                json_str = response[start_idx:end_idx + 1]
                
                # Clean common issues
                json_str = json_str.replace("'", '"')
                json_str = re.sub(r',\s*}', '}', json_str)
                json_str = re.sub(r',\s*]', ']', json_str)
                json_str = re.sub(r'"\s*\n\s*"', '" "', json_str)  # Fix broken strings
                
                data = json.loads(json_str)
                
                raw_tags = data.get("tags", [])
                feedback.tags = self._validate_tags(raw_tags, metrics)
                feedback.explanation = str(data.get("explanation", ""))[:500]
                feedback.suggestion = str(data.get("suggestion", ""))[:400]
                feedback.parse_success = True
            else:
                feedback = self._generate_fallback_feedback(response, metrics)
                
        except (json.JSONDecodeError, KeyError, TypeError):
            feedback = self._generate_fallback_feedback(response, metrics)
        
        return feedback
    
    def _validate_tags(self, tags: List[str], metrics: MetricScores = None) -> List[str]:
        """Validate and fix tag conflicts."""
        if not tags:
            if metrics:
                tags, _ = self._assign_tags(metrics)
            else:
                tags = ["Partially Correct"]
        
        # Normalize tags
        valid_tags = []
        for tag in tags:
            if isinstance(tag, str):
                # Find matching tag (case-insensitive)
                for valid in self.ALL_TAGS:
                    if tag.lower().strip() == valid.lower():
                        valid_tags.append(valid)
                        break
        
        if not valid_tags:
            if metrics:
                return self._assign_tags(metrics)[0]
            return ["Partially Correct"]
        
        # If Correct, remove all issue tags
        if "Correct" in valid_tags:
            return ["Correct"]
        
        # Ensure exactly one correctness tag
        correctness_in_tags = [t for t in valid_tags if t in self.CORRECTNESS_TAGS]
        issue_tags = [t for t in valid_tags if t in self.ISSUE_TAGS]
        # Ensure exactly one correctness tag
        # Deduplicate issue tags
        issue_tags = list(dict.fromkeys(issue_tags))
        
        if len(correctness_in_tags) == 0:
            if any(t in ["Logical Error", "Factual Error", "Off-Topic"] for t in issue_tags):
                return ["Incorrect"] + issue_tags
            else:
                return ["Partially Correct"] + issue_tags
        elif len(correctness_in_tags) > 1:
            if "Incorrect" in correctness_in_tags:
                return ["Incorrect"] + issue_tags
            else:
                return ["Partially Correct"] + issue_tags
        
        return [correctness_in_tags[0]] + issue_tags
    
    def _generate_fallback_feedback(self, raw_response: str, metrics: MetricScores = None) -> LLMFeedback:
        """Generate high-quality rule-based feedback with SPECIFIC suggestions."""
        feedback = LLMFeedback(raw_response=raw_response, parse_success=False)
        
        if metrics is None:
            feedback.explanation = raw_response[:400] if raw_response else "Unable to evaluate."
            feedback.tags = ["Partially Correct"]
            return feedback
        
        tags, correctness = self._assign_tags(metrics)
        explanations = []
        suggestions = []
        
        if correctness == "Correct":
            explanations.append("Excellent answer that demonstrates strong understanding of the topic.")
            suggestions.append("Great work! Keep up the quality.")
        else:
            if "Off-Topic" in tags:
                explanations.append(f"The answer does not address the question (semantic similarity: {metrics.semantic_score:.0%}).")
                suggestions.append("Please re-read the question carefully and provide a response that directly addresses what is being asked.")
            
            if "Logical Error" in tags:
                explanations.append("The answer contains statements that contradict the reference material.")
                suggestions.append("Check your understanding of the core concepts. Make sure your statements align with established facts.")
            
            if "Factual Error" in tags and "Logical Error" not in tags:
                explanations.append("The answer contains factual inaccuracies.")
                suggestions.append("Review the source material and verify your facts before answering.")
            
            if "Missing Concepts" in tags and metrics.missing_keywords:
                missing_str = ", ".join(metrics.missing_keywords[:3])
                explanations.append(f"Key concepts are missing from your answer: {missing_str}.")
                suggestions.append(f"Include these important terms in your answer: '{missing_str}'. Explain how they relate to the topic.")
            
            if "Vague Expression" in tags:
                explanations.append("Your answer is on topic but too general.")
                suggestions.append("Be more specific. Instead of general statements, use precise terms and provide concrete examples or details.")
            
            if "Grammar Error" in tags:
                explanations.append(f"Writing quality needs improvement (grammar score: {metrics.grammar_score:.0%}).")
                suggestions.append("Proofread your answer. Check for complete sentences, proper capitalization, and correct verb forms.")
            
            # Default if no specific issues
            if len(explanations) == 0:
                explanations.append("The answer is partially correct but could be improved.")
                suggestions.append("Expand your answer with more details and make sure to cover all key points from the topic.")
        
        feedback.tags = tags
        feedback.explanation = " ".join(explanations)
        feedback.suggestion = " ".join(suggestions)
        
        return feedback
    
    def generate_feedback(self,
                          context: str,
                          question: str,
                          reference: str,
                          student: str,
                          metrics: MetricScores) -> LLMFeedback:
        """Generate feedback using Qwen2 with optimized generation."""
        prompt = self._build_prompt(context, question, reference, student, metrics)
        
        with torch.no_grad():
            inputs = self.reasoning_tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=800
            ).to(self.device)
            
            outputs = self.reasoning_model.generate(
                **inputs,
                max_new_tokens=500,
                do_sample=False,
                num_beams=1,
                pad_token_id=self.reasoning_tokenizer.eos_token_id
            )
            
            response = self.reasoning_tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            )
        
        return self._parse_llm_response(response, metrics)
    
    # =========================================================================
    # MAIN GRADING METHOD
    # =========================================================================
    
    def grade_answer(
        self,
        context: str,
        question: str,
        reference: str,
        student: str,
        weights: Dict[str, float] = None
    ) -> GradingResult:
        """
        Grade a student answer against a reference answer.
        
        Args:
            context: Background context for the question
            question: The question being answered
            reference: The reference/expected answer
            student: The student's answer
            weights: Optional custom weights for final grade calculation.
                     Keys: 'semantic', 'coverage', 'formality', 'grammar', 'logic'
                     Values should sum to 1.0. Default is 20% for each criterion.
                     Example: {"semantic": 0.30, "coverage": 0.25, "formality": 0.10, 
                              "grammar": 0.15, "logic": 0.20}
        
        Returns:
            GradingResult with metrics (including final_grade on 0-100 scale) and feedback
        """
        result = GradingResult()
        
        if not student or not student.strip():
            result.is_valid = False
            result.skip_reason = "Empty student answer"
            return result
        
        word_count = len(student.split())
        
        if word_count < self.MIN_WORDS_FOR_FULL_ANALYSIS:
            result.metrics.semantic_score = self.get_simple_word_match_score(reference, student)
            result.metrics.coverage_score = result.metrics.semantic_score
            result.metrics.formality_score = 0.5
            result.metrics.grammar_score = 0.5
            result.metrics.logic_score = result.metrics.semantic_score
            # Calculate final grade for short answers
            result.metrics.final_grade = calculate_final_grade(result.metrics, weights)
            
            result.feedback = LLMFeedback(
                tags=["Incomplete", "Vague Expression"],
                explanation=f"Answer too short ({word_count} words). Cannot fully evaluate.",
                suggestion="Please provide a more detailed and complete answer.",
                parse_success=True
            )
            return result
        
        self._log("\nCalculating metrics...")
        
        self._log("   - Semantic similarity...")
        result.metrics.semantic_score = self.get_semantic_score(reference, student)
        
        self._log("   - Keyword coverage...")
        coverage, missing = self.get_keyword_coverage(question, reference, student)
        result.metrics.coverage_score = coverage
        result.metrics.missing_keywords = missing
        
        self._log("   - Formality...")
        result.metrics.formality_score = self.get_formality_score(student)
        
        self._log("   - Grammar...")
        result.metrics.grammar_score = self.get_grammar_score(student)
        
        self._log("   - Logic/coherence...")
        # Pass grammar score to logic calculation
        logic_score, logic_details = self.get_logic_score(
            reference, student, result.metrics.grammar_score
        )
        result.metrics.logic_score = logic_score
        result.metrics.logic_details = logic_details
        
        self._log("   - Final grade...")
        result.metrics.final_grade = calculate_final_grade(result.metrics, weights)
        
        self._log("\nGenerating feedback with Qwen2...")
        
        result.feedback = self.generate_feedback(
            context, question, reference, student, result.metrics
        )
        
        self._log("Grading complete!\n")
        
        return result
    
    def grade_batch(self, items: List[Dict[str, str]], weights: Dict[str, float] = None) -> List[GradingResult]:
        """
        Grade multiple answers.
        
        Args:
            items: List of dicts with 'context', 'question', 'reference', 'student' keys
            weights: Optional custom weights for final grade calculation (applied to all items)
        
        Returns:
            List of GradingResult objects
        """
        results = []
        total = len(items)
        
        for i, item in enumerate(items, 1):
            self._log(f"\n{'='*60}")
            self._log(f"Grading answer {i}/{total}")
            self._log(f"{'='*60}")
            
            result = self.grade_answer(
                context=item.get("context", ""),
                question=item["question"],
                reference=item["reference"],
                student=item["student"],
                weights=weights
            )
            results.append(result)
        
        return results
    
    
    def get_available_tags(self) -> Dict[str, List[str]]:
        """Return available tags for reference."""
        return {
            "correctness_tags": self.CORRECTNESS_TAGS,
            "issue_tags": self.ISSUE_TAGS,
            "all_tags": self.ALL_TAGS
        }
    
    def get_thresholds(self) -> Dict[str, float]:
        """Return current thresholds for debugging."""
        return self.THRESHOLDS

## 5. Demo & Testing

Let's test the grader with some sample data.

In [28]:
# Initialize the grader
print("Initializing HybridASAGGrader...")
print("This may take a few minutes to download and load all models.\n")

grader = HybridASAGGrader(device=DEVICE, verbose=True)

Initializing HybridASAGGrader...
This may take a few minutes to download and load all models.


Loading Models...

[1/6] Loading Semantic Model (SimCSE)...


No sentence-transformers model found with name princeton-nlp/sup-simcse-roberta-large. Creating a new one with mean pooling.


      Done: princeton-nlp/sup-simcse-roberta-large

[2/6] Loading KeyBERT Model...
      Done: all-MiniLM-L6-v2 (KeyBERT backend)

[3/6] Loading Formality Model...
      Done: cointegrated/roberta-base-formality

[4/6] Loading Grammar Model...


Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


      Done: textattack/roberta-base-CoLA

[5/6] Loading Logic Model (NLI)...
      Done: cross-encoder/nli-deberta-v3-base

[6/6] Loading Reasoning Model (Qwen2.5-3B-Instruct)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.16s/it]


      Done: Qwen/Qwen2.5-3B-Instruct

All models loaded successfully!



In [29]:
# Sample test data with EXPECTED outcomes
test_cases = [
    {
        "name": "Good Answer",
        "expected_correctness": "Correct",
        "expected_issues": [],
        "context": "Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water. It generally involves the green pigment chlorophyll and generates oxygen as a byproduct.",
        "question": "Explain the process of photosynthesis and its importance.",
        "reference": "Photosynthesis is a biological process where plants convert sunlight, carbon dioxide, and water into glucose and oxygen using chlorophyll. This process is essential for producing food for plants and releasing oxygen into the atmosphere, which is vital for most living organisms.",
        "student": "Photosynthesis is when plants use sunlight, CO2, and water to make glucose and oxygen. Chlorophyll helps capture the light energy. This process is important because it provides food for plants and oxygen for animals to breathe."
    },
    {
        "name": "Partial Answer (Missing Keywords)",
        "expected_correctness": "Partially Correct",
        "expected_issues": ["Missing Concepts", "Vague Expression"],
        "context": "Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water.",
        "question": "Explain the process of photosynthesis and its importance.",
        "reference": "Photosynthesis is a biological process where plants convert sunlight, carbon dioxide, and water into glucose and oxygen using chlorophyll. This process is essential for producing food for plants and releasing oxygen into the atmosphere.",
        "student": "Plants make their own food using light. This is important for the environment."
    },
    {
        "name": "Incorrect Answer",
        "expected_correctness": "Incorrect",
        "expected_issues": ["Logical Error", "Factual Error"],
        "context": "Photosynthesis is the process by which green plants use sunlight to synthesize foods.",
        "question": "Explain the process of photosynthesis.",
        "reference": "Photosynthesis is a biological process where plants convert sunlight, carbon dioxide, and water into glucose and oxygen using chlorophyll.",
        "student": "Photosynthesis is when animals eat plants to get energy. The plants die and become fertilizer for other plants."
    },
    {
        "name": "Grammar Issues",
        "expected_correctness": "Partially Correct",
        "expected_issues": ["Grammar Error", "Vague Expression"],
        "context": "The water cycle describes how water evaporates from the surface of the earth.",
        "question": "Describe the water cycle.",
        "reference": "The water cycle is the continuous movement of water through evaporation, condensation, and precipitation.",
        "student": "water cycle is when water go up to sky and then it rain down again and again this happen many time"
    },
    {
        "name": "Very Short Answer",
        "expected_correctness": "Incomplete",
        "expected_issues": ["Incomplete", "Vague Expression"],
        "context": "The water cycle describes continuous movement of water.",
        "question": "Describe the water cycle.",
        "reference": "The water cycle involves evaporation, condensation, and precipitation.",
        "student": "water evaporation"
    }
]

In [30]:
def display_result(test_case: dict, result: GradingResult):
    """Pretty print a grading result with expected vs actual comparison."""
    name = test_case["name"]
    expected_correctness = test_case.get("expected_correctness", "Unknown")
    expected_issues = test_case.get("expected_issues", [])
    
    print(f"\n{'='*70}")
    print(f"Test Case: {name}")
    print(f"{'='*70}")
    
    if not result.is_valid:
        print(f"\nSkipped: {result.skip_reason}")
        return
    
    # Metrics
    print("\nMETRICS:")
    print(f"   - Semantic Score:    {result.metrics.semantic_score:.3f}")
    print(f"   - Coverage Score:    {result.metrics.coverage_score:.3f}")
    if result.metrics.missing_keywords:
        print(f"     Missing:           {', '.join(result.metrics.missing_keywords[:5])}")
    print(f"   - Formality Score:   {result.metrics.formality_score:.3f}")
    print(f"   - Grammar Score:     {result.metrics.grammar_score:.3f}")
    print(f"   - Logic Score:       {result.metrics.logic_score:.3f}")
    if result.metrics.logic_details:
        print(f"     Entailment:        {result.metrics.logic_details.get('entailment', 0):.3f}")
        print(f"     Contradiction:     {result.metrics.logic_details.get('contradiction', 0):.3f}")
    
    # Tags comparison
    actual_tags = result.feedback.tags
    actual_correctness = actual_tags[0] if actual_tags else "Unknown"
    actual_issues = actual_tags[1:] if len(actual_tags) > 1 else []
    
    print("\nTAGS COMPARISON:")
    print(f"   Expected Correctness: {expected_correctness}")
    print(f"   Actual Correctness:   {actual_correctness}", end="")
    if actual_correctness == expected_correctness:
        print(" [MATCH]")
    else:
        print(" [MISMATCH]")
    
    print(f"   Expected Issues:      {expected_issues}")
    print(f"   Actual Issues:        {actual_issues}", end="")
    # Check if issues match (order doesn't matter)
    expected_set = set(expected_issues)
    actual_set = set(actual_issues)
    if expected_set == actual_set:
        print(" [MATCH]")
    elif expected_set.issubset(actual_set) or actual_set.issubset(expected_set):
        print(" [PARTIAL]")
    else:
        print(" [MISMATCH]")
    
    # Feedback
    print("\nLLM FEEDBACK:")
    print(f"   - Parse Success:     {result.feedback.parse_success}")
    print(f"   - All Tags:          {result.feedback.tags}")
    print(f"   - Explanation:       {result.feedback.explanation[:100]}..." if len(result.feedback.explanation) > 100 else f"   - Explanation:       {result.feedback.explanation}")
    print(f"   - Suggestion:        {result.feedback.suggestion[:100]}..." if len(result.feedback.suggestion) > 100 else f"   - Suggestion:        {result.feedback.suggestion}")


def evaluate_grading_accuracy(test_cases: list, results: list) -> dict:
    """Calculate accuracy metrics for grading."""
    total = len(test_cases)
    correctness_matches = 0
    issue_matches = 0
    partial_issue_matches = 0
    
    for test, result in zip(test_cases, results):
        if not result.is_valid:
            continue
            
        expected_correctness = test.get("expected_correctness", "")
        expected_issues = set(test.get("expected_issues", []))
        
        actual_tags = result.feedback.tags
        actual_correctness = actual_tags[0] if actual_tags else ""
        actual_issues = set(actual_tags[1:]) if len(actual_tags) > 1 else set()
        
        if actual_correctness == expected_correctness:
            correctness_matches += 1
        
        if expected_issues == actual_issues:
            issue_matches += 1
        elif expected_issues.issubset(actual_issues) or actual_issues.issubset(expected_issues):
            partial_issue_matches += 1
    
    return {
        "total_cases": total,
        "correctness_accuracy": correctness_matches / total * 100,
        "exact_issue_match": issue_matches / total * 100,
        "partial_issue_match": (issue_matches + partial_issue_matches) / total * 100
    }

In [31]:
# Run all test cases and evaluate
print("Starting grading tests...")
print("="*70)

all_results = []
for test in test_cases:
    result = grader.grade_answer(
        context=test["context"],
        question=test["question"],
        reference=test["reference"],
        student=test["student"]
    )
    all_results.append(result)
    display_result(test, result)

# Calculate and display accuracy
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

accuracy = evaluate_grading_accuracy(test_cases, all_results)
print(f"\n   Total Test Cases:        {accuracy['total_cases']}")
print(f"   Correctness Accuracy:    {accuracy['correctness_accuracy']:.1f}%")
print(f"   Exact Issue Match:       {accuracy['exact_issue_match']:.1f}%")
print(f"   Partial+ Issue Match:    {accuracy['partial_issue_match']:.1f}%")

print("\n" + "="*70)
print("All tests completed!")
print("="*70)

# Show available tags for reference
print("\nAvailable Tag System:")
tags_info = grader.get_available_tags()
print(f"   Correctness Tags: {tags_info['correctness_tags']}")
print(f"   Issue Tags:       {tags_info['issue_tags']}")

Starting grading tests...

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Test Case: Good Answer

METRICS:
   - Semantic Score:    0.964
   - Coverage Score:    0.800
     Missing:           convert
   - Formality Score:   1.000
   - Grammar Score:     0.973
   - Logic Score:       0.702
     Entailment:        0.015
     Contradiction:     0.000

TAGS COMPARISON:
   Expected Correctness: Correct
   Actual Correctness:   Correct [MATCH]
   Expected Issues:      []
   Actual Issues:        [] [MATCH]

LLM FEEDBACK:
   - Parse Success:     True
   - All Tags:          ['Correct']
   - Explanation:       The student's answer correctly describes the process of photosynthesis by mentioning that plants use...
   - Suggestion:        To improve, the student should include the word 'convert' to accurately describe the process. Also, .

In [9]:
# Quick summary of results
print("\n" + "="*60)
print("FINAL OPTIMIZED RESULTS")
print("="*60)

correct_count = 0
for i, (test, result) in enumerate(zip(test_cases, all_results)):
    expected_correct = test["expected_correctness"]
    actual_tags = result.feedback.tags if result.feedback else []
    actual_correct = actual_tags[0] if actual_tags else "Unknown"
    
    is_match = expected_correct == actual_correct
    if is_match:
        correct_count += 1
    match = "[MATCH]" if is_match else "[MISMATCH]"
    print(f"\n{i+1}. {test['name']}")
    print(f"   Expected: {expected_correct} | Actual: {actual_correct} {match}")
    print(f"   Metrics: sem={result.metrics.semantic_score:.2f}, cov={result.metrics.coverage_score:.2f}, gram={result.metrics.grammar_score:.2f}, log={result.metrics.logic_score:.2f}")
    print(f"   Tags: {actual_tags}")

print("\n" + "="*60)
total = len(test_cases)
print(f"Correctness Accuracy: {correct_count}/{total} = {correct_count/total:.0%}")
print(f"Partial+ Issue Match: {accuracy['partial_issue_match']:.0f}%")
print("="*60)


FINAL OPTIMIZED RESULTS

1. Good Answer
   Expected: Correct | Actual: Partially Correct [MISMATCH]
   Metrics: sem=0.96, cov=0.80, gram=0.97, log=0.70
   Tags: ['Partially Correct', 'Missing Concepts']

2. Partial Answer (Missing Keywords)
   Expected: Partially Correct | Actual: Partially Correct [MATCH]
   Metrics: sem=0.76, cov=0.40, gram=0.97, log=0.72
   Tags: ['Partially Correct', 'Missing Concepts']

3. Incorrect Answer
   Expected: Incorrect | Actual: Incorrect [MATCH]
   Metrics: sem=0.78, cov=0.40, gram=0.97, log=0.00
   Tags: ['Incorrect', 'Logical Error', 'Factual Error', 'Missing Concepts']

4. Grammar Issues
   Expected: Partially Correct | Actual: Partially Correct [MATCH]
   Metrics: sem=0.80, cov=1.00, gram=0.26, log=0.56
   Tags: ['Partially Correct', 'Grammar Error']

5. Very Short Answer
   Expected: Incomplete | Actual: Incomplete [MATCH]
   Metrics: sem=0.12, cov=0.12, gram=0.50, log=0.12
   Tags: ['Incomplete', 'Vague Expression']

Correctness Accuracy: 4/5 = 8

## 6. Interactive Grading

Use this cell for interactive testing with your own data.

In [10]:
# Interactive grading example
# Modify these values to test with your own data

context = "Machine learning is a subset of artificial intelligence that enables systems to learn from data."

question = "What is machine learning and how does it relate to AI?"

reference = "Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for each task."

student = "Machine learning is part of AI where computers learn from data to make predictions."

# Grade the answer
result = grader.grade_answer(context, question, reference, student)

# Create test case dict for display
interactive_test = {
    "name": "Interactive Test",
    "expected_correctness": "Unknown",  # Set your expectation here
    "expected_issues": [],
    "context": context,
    "question": question,
    "reference": reference,
    "student": student
}

# Display results
display_result(interactive_test, result)

# Get result as dictionary (for JSON export)
print("\nResult as Dictionary:")
print(json.dumps(result.to_dict(), indent=2, ensure_ascii=False))


Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Test Case: Interactive Test

METRICS:
   - Semantic Score:    0.927
   - Coverage Score:    0.750
     Missing:           patterns
   - Formality Score:   1.000
   - Grammar Score:     0.929
   - Logic Score:       0.700
     Entailment:        0.998
     Contradiction:     0.000

TAGS COMPARISON:
   Expected Correctness: Unknown
   Actual Correctness:   Partially Correct [MISMATCH]
   Expected Issues:      []
   Actual Issues:        ['Missing Concepts'] [PARTIAL]

LLM FEEDBACK:
   - Parse Success:     True
   - All Tags:          ['Partially Correct', 'Missing Concepts']
   - Explanation:       The student's answer correctly identifies that machine learning is a part of AI and mentions that co...
   - Suggestion:        To improve, the student should incorporate the term 'patte

## 7. Export Results

Example of exporting grading results to JSON.

In [32]:
def export_results_to_json(results: List[GradingResult], filepath: str):
    """Export grading results to a JSON file."""
    export_data = [result.to_dict() for result in results]
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"Results exported to: {filepath}")

# Example: Grade batch and export
batch_items = [
    {
        "context": test["context"],
        "question": test["question"],
        "reference": test["reference"],
        "student": test["student"]
    }
    for test in test_cases  # First 3 test cases
]

batch_results = grader.grade_batch(batch_items)
export_results_to_json(batch_results, "grading_results.json")


Grading answer 1/5

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Grading answer 2/5

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Grading answer 3/5

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Grading answer 4/5

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


Grading answer 5/5
Results exported to: grading_results.json


## 8. Final Grade Calculation

The system calculates a **final_grade** on a 0-100 scale based on 5 criteria with configurable weights.

In [12]:
# Demo: Final Grade Calculation with Custom Weights
# =================================================

# Example 1: Using default weights (20% each criterion)
print("=" * 60)
print("FINAL GRADE DEMO")
print("=" * 60)

# Show default weights
print("\nDEFAULT WEIGHTS (20% each):")
for criterion, weight in DEFAULT_WEIGHTS.items():
    print(f"   • {criterion.capitalize():12}: {weight*100:.0f}%")

# Grade with default weights
print("\n" + "-" * 40)
print("Grading with DEFAULT weights...")
result_default = grader.grade_answer(
    context="Photosynthesis is the process by which plants make food.",
    question="Explain photosynthesis.",
    reference="Photosynthesis is the process where plants convert sunlight, CO2, and water into glucose and oxygen using chlorophyll.",
    student="Plants use sunlight and water to make food and release oxygen."
)

print(f"\nMETRICS:")
print(f"   • Semantic Score:   {result_default.metrics.semantic_score:.2%}")
print(f"   • Coverage Score:   {result_default.metrics.coverage_score:.2%}")
print(f"   • Formality Score:  {result_default.metrics.formality_score:.2%}")
print(f"   • Grammar Score:    {result_default.metrics.grammar_score:.2%}")
print(f"   • Logic Score:      {result_default.metrics.logic_score:.2%}")
print(f"\nFINAL GRADE (default weights): {result_default.metrics.final_grade}/100")

# Example 2: Using custom weights (prioritize semantic and coverage)
print("\n" + "=" * 60)
print("Grading with CUSTOM weights...")

custom_weights = {
    "semantic": 0.35,   # 35% - prioritize meaning
    "coverage": 0.25,   # 25% - emphasize key concepts
    "formality": 0.10,  # 10% - less strict on formality
    "grammar": 0.15,    # 15% - moderate grammar importance
    "logic": 0.15       # 15% - moderate logic importance
}

print("\nCUSTOM WEIGHTS:")
for criterion, weight in custom_weights.items():
    print(f"   • {criterion.capitalize():12}: {weight*100:.0f}%")

# Grade with custom weights
result_custom = grader.grade_answer(
    context="Photosynthesis is the process by which plants make food.",
    question="Explain photosynthesis.",
    reference="Photosynthesis is the process where plants convert sunlight, CO2, and water into glucose and oxygen using chlorophyll.",
    student="Plants use sunlight and water to make food and release oxygen.",
    weights=custom_weights
)

print(f"\nFINAL GRADE (custom weights): {result_custom.metrics.final_grade}/100")

# Comparison
print("\n" + "=" * 60)
print("COMPARISON:")
print(f"   Default weights (20% each):     {result_default.metrics.final_grade}/100")
print(f"   Custom weights (semantic-heavy): {result_custom.metrics.final_grade}/100")
print(f"   Difference:                      {abs(result_custom.metrics.final_grade - result_default.metrics.final_grade):.2f} points")

FINAL GRADE DEMO

DEFAULT WEIGHTS (20% each):
   • Semantic    : 20%
   • Coverage    : 20%
   • Formality   : 20%
   • Grammar     : 20%
   • Logic       : 20%

----------------------------------------
Grading with DEFAULT weights...

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...
Grading complete!


METRICS:
   • Semantic Score:   84.43%
   • Coverage Score:   80.00%
   • Formality Score:  99.99%
   • Grammar Score:    97.51%
   • Logic Score:      69.87%

FINAL GRADE (default weights): 86.36/100

Grading with CUSTOM weights...

CUSTOM WEIGHTS:
   • Semantic    : 35%
   • Coverage    : 25%
   • Formality   : 10%
   • Grammar     : 15%
   • Logic       : 15%

Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...
   - Final grade...

Generating feedback with Qwen2...


In [13]:
# Example 3: Recalculate final grade with different weights (without re-grading)
# Useful for experimenting with weight configurations

print("\n" + "=" * 60)
print("RECALCULATING FINAL GRADE WITH DIFFERENT WEIGHT PRESETS")
print("=" * 60)

# Define different weight presets
weight_presets = {
    "Balanced (Default)": {
        "semantic": 0.20, "coverage": 0.20, "formality": 0.20, 
        "grammar": 0.20, "logic": 0.20
    },
    "Content-Focused": {
        "semantic": 0.40, "coverage": 0.30, "formality": 0.05, 
        "grammar": 0.10, "logic": 0.15
    },
    "Academic Writing": {
        "semantic": 0.20, "coverage": 0.15, "formality": 0.25, 
        "grammar": 0.25, "logic": 0.15
    },
    "Logic-Heavy": {
        "semantic": 0.25, "coverage": 0.15, "formality": 0.10, 
        "grammar": 0.15, "logic": 0.35
    }
}

# Use existing metrics from result_default
metrics = result_default.metrics

print(f"\n📈 Fixed Metrics (from previous grading):")
print(f"   Semantic: {metrics.semantic_score:.2%} | Coverage: {metrics.coverage_score:.2%}")
print(f"   Formality: {metrics.formality_score:.2%} | Grammar: {metrics.grammar_score:.2%}")
print(f"   Logic: {metrics.logic_score:.2%}")

print(f"\n🎯 Final Grades with Different Weight Presets:")
print("-" * 50)

for preset_name, weights in weight_presets.items():
    final_grade = calculate_final_grade(metrics, weights)
    print(f"   {preset_name:20} → {final_grade:6.2f}/100")


RECALCULATING FINAL GRADE WITH DIFFERENT WEIGHT PRESETS

📈 Fixed Metrics (from previous grading):
   Semantic: 84.43% | Coverage: 80.00%
   Formality: 99.99% | Grammar: 97.51%
   Logic: 69.87%

🎯 Final Grades with Different Weight Presets:
--------------------------------------------------
   Balanced (Default)   →  86.36/100
   Content-Focused      →  83.00/100
   Academic Writing     →  88.74/100
   Logic-Heavy          →  82.19/100


---

## Summary

This notebook implements a Hybrid ASAG System with:

### Feature Extraction Layer
| Metric | Model | Purpose |
|--------|-------|----------|
| Semantic | SimCSE (RoBERTa-Large) | Measures meaning similarity |
| Keywords | KeyBERT + Semantic Matching | Checks concept coverage |
| Formality | RoBERTa-base-formality | Academic language check |
| Grammar | RoBERTa-base-CoLA | Grammatical correctness |
| Logic | DeBERTa-v3 NLI | Logical coherence |

### Reasoning Layer
| Model | Purpose |
|-------|----------|
| Qwen2.5-3B-Instruct | Generate detailed feedback with tags, explanations, and suggestions |

### Final Grade Calculation (0-100 Scale)
The system calculates a weighted **final_grade** based on 5 criteria:

| Criterion | Default Weight | Description |
|-----------|----------------|-------------|
| Semantic | 20% | Meaning similarity to reference answer |
| Coverage | 20% | Key concepts/keywords coverage |
| Formality | 20% | Academic writing style |
| Grammar | 20% | Grammatical correctness |
| Logic | 20% | Logical coherence with reference |

**Custom Weights Example:**
```python
# Prioritize content over style
custom_weights = {
    "semantic": 0.35,
    "coverage": 0.25,
    "formality": 0.10,
    "grammar": 0.15,
    "logic": 0.15
}

result = grader.grade_answer(
    context="...", question="...", 
    reference="...", student="...",
    weights=custom_weights
)

print(f"Final Grade: {result.metrics.final_grade}/100")
```

### Key Features
- MPS (Apple Silicon) optimized
- Semantic keyword matching
- Dynamic keyword extraction
- **Configurable final grade weights**
- Robust JSON parsing
- Short answer handling
- Batch processing support
- JSON export capability